# Fase 4: Augmentação por Simetria

Gera 3 variantes simétricas de cada NPZ original com sufixo no nome do arquivo:
- `dataset_pequeno_NNNN_refH.npz` - imagem espelhada/reflexo horizontal de cada estado do tabuleiro
- `dataset_pequeno_NNNN_refV.npz` - imagem espelhada/reflexo vertical de cada estado do tabuleiro
- `dataset_pequeno_NNNN_r180.npz` - rotação 180° de cada estado do tabuleiro

In [2]:
import os
import sys
import glob
import time
import numpy as np

ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# Modulo de permutacoes de simetria.
from permutacoes_simetria_pontinhos import aplicar_simetria

print(f'Backend root = {ROOT}')
print('Imports OK.')

Backend root = d:\Pos_Graduacao\Trabalho_Final\gerador_dados
Imports OK.


In [3]:
# Configuração
DIR_NPZ = os.path.join(ROOT, 'dados', 'base_limpa')

# Mapeamento sym_id: sufixo do arquivo gerado.
# sym_id=0: identidade (não gera arquivo)
# sym_id=1: reflexão horizontal
# sym_id=2: reflexão vertical
# sym_id=3: rotação 180°
SIMETRIAS = [
    (1, '_refH'),
    (2, '_refV'),
    (3, '_r180'),
]

print(f'DIR_NPZ  = {DIR_NPZ}')
print(f'Simetrias: {[(s, sf) for s, sf in SIMETRIAS]}')

DIR_NPZ  = d:\Pos_Graduacao\Trabalho_Final\gerador_dados\dados\base_limpa
Simetrias: [(1, '_refH'), (2, '_refV'), (3, '_r180')]


## Passo 1: Apagar todos os arquivos de simetria (caso existam)

In [4]:
sufixos = [sf for _, sf in SIMETRIAS]
a_deletar = [
    f for f in glob.glob(os.path.join(DIR_NPZ, 'dataset_pequeno_*.npz'))
    if any(os.path.splitext(os.path.basename(f))[0].endswith(s) for s in sufixos)
]

print(f'Arquivos sufixados a deletar: {len(a_deletar)}')
for f in a_deletar:
    os.remove(f)
print('Deletados. Pronto para regenerar.')

Arquivos sufixados a deletar: 0
Deletados. Pronto para regenerar.


## Passo 2: Descobrir NPZs originais

In [5]:
SUFIXOS_SIMETRIA = tuple(sf for _, sf in SIMETRIAS)
arquivos_originais = sorted([
    f for f in glob.glob(os.path.join(DIR_NPZ, 'dataset_pequeno_*.npz'))
    if not any(os.path.splitext(os.path.basename(f))[0].endswith(s) for s in SUFIXOS_SIMETRIA)
])

print(f'NPZs originais: {len(arquivos_originais)}')

NPZs originais: 105


## Passo 3: Gerar variantes

In [6]:
def gerar_variante(caminho_orig: str, sym_id: int, sufixo: str) -> dict:
    """Aplica simetria sym_id ao NPZ e salva com sufixo. Sobrescrita atômica."""
    t0 = time.time()
    with np.load(caminho_orig, allow_pickle=False) as f:
        d = dict(f)

    d_sim = aplicar_simetria(d, sym_id)

    base, ext = os.path.splitext(caminho_orig)
    caminho_saida = base + sufixo + ext
    tmp = caminho_saida + '.tmp.npz'
    np.savez_compressed(tmp, **d_sim)
    os.replace(tmp, caminho_saida)

    return {'origem': caminho_orig, 'destino': caminho_saida, 'tempo_s': time.time() - t0}

relatorio = []
t_inicio = time.time()
for arq in arquivos_originais:
    for sym_id, sufixo in SIMETRIAS:
        info = gerar_variante(arq, sym_id, sufixo)
        relatorio.append(info)

print(f'Variantes geradas : {len(relatorio)}')
print(f'Tempo total       : {time.time() - t_inicio:.1f}s')

Variantes geradas : 315
Tempo total       : 69.1s


## Auditoria final

In [7]:
todos = sorted(glob.glob(os.path.join(DIR_NPZ, 'dataset_pequeno_*.npz')))
n_total = len(todos)
n_originais = sum(
    1 for f in todos
    if not any(os.path.splitext(os.path.basename(f))[0].endswith(s) for s in SUFIXOS_SIMETRIA)
)
n_sufixados = n_total - n_originais

print(f'Originais : {n_originais}')
print(f'Sufixados : {n_sufixados}  (esperado {n_originais * len(SIMETRIAS)})')
print(f'Total     : {n_total}  (esperado {n_originais * (1 + len(SIMETRIAS))})')

# Verificar que cada original tem os 3 sufixos.
originais = [f for f in todos
             if not any(os.path.splitext(os.path.basename(f))[0].endswith(s)
                        for s in SUFIXOS_SIMETRIA)]
erros = []
for orig in originais:
    base = os.path.splitext(orig)[0]
    for _, sf in SIMETRIAS:
        esperado = base + sf + '.npz'
        if not os.path.exists(esperado):
            erros.append(f'Faltando: {os.path.basename(esperado)}')

if erros:
    print(f'FALHA: {len(erros)} arquivo(s) ausente(s):')
    for e in erros[:10]:
        print(f'  {e}')
else:
    print(f'OK: {n_total} NPZs ({n_originais} originais × 4 = {n_originais * 4} esperados).')
    print('Pronto para Treinamento_CNN_Pontinhos_V8.ipynb.')

Originais : 105
Sufixados : 315  (esperado 315)
Total     : 420  (esperado 420)
OK: 420 NPZs (105 originais × 4 = 420 esperados).
Pronto para Treinamento_CNN_Pontinhos_V8.ipynb.
